### Test of FOD python code

these are snippets of code for chunks of code taken from new_fod.py code 
in working FOD to reverse-engineer for refactoring


TODO: this is not currently working - need to refactor all configuration which uses the old system (NARR_INPUT = narr_grid_latlon etc)

In [1]:
%reload_ext autoreload
%autoreload 2
import importlib


In [2]:
# standard imports 
import sys
import os
from datetime import datetime

# numerical py imports for typing
import numpy as np

In [3]:
# imports needs for response
import base64  ### only needed for KML (I think)
import json

### Main Program Imports
allow reloading for dev

In [4]:
# to allow for reloading, have to load the module by name or specific functions
# stupid stupid python path is stupid

sys.path.insert(0, os.path.abspath('../../src'))
# load this once
from mioffset import fod3
from mioffset import narr_data


MIOFFSET DEVELOPMENT VERSION - NOT FOR PRODUCTION USE


### Our Module imports

use reload here for dev

In [5]:

# now reload them as needed
importlib.reload(fod3)
importlib.reload(narr_data)

from mioffset.fod3 import fod_model, matplotlib_to_svg, setback_text_table,fod_plot_to_ll, fod_kml, fod_geojson, fod2dict, footprint_plots
from mioffset.narr_data import GridIndex, GridIndexS3, wind_data_factory, WindData, WindDataS3, filter_narr_timeseries
from mioffset.aws import get_aws_config, get_s3_client, S3Client

MIOFFSET DEVELOPMENT VERSION - NOT FOR PRODUCTION USE


## Input variables

This include a timestamp variable formatted with His in PHP
 - 24-hour format of an hour with leading zeros
 - Minutes with leading zeros 	00 to 59
 - Seconds with leading zeros 	00 through 59
 - milli seconds

In [6]:

# original code

# latval = float(sys.argv[1])
# lonval = float(sys.argv[2])
# odor_index = float(sys.argv[3])
# time_stamp = sys.argv[4]

# some coordinates I just clicked on 
# 43.14319° N, 84.23689° W

import os
from dotenv import load_dotenv
load_dotenv()
lat:float = float(43.14319)
lon:float = float(-84.23689)
odor_index:int = 10
time_stamp = datetime.now().strftime("%H:%M:%S.%f")
time_flag:str = os.getenv("TIME_FLAG","")
narr_bucket:str = os.getenv("NARR_BUCKET","")
narr_data_dir:str = os.getenv("NARR_DATA_DIR","")
narr_grid_file:str = os.getenv("NARR_GRID_LATLON_S3", "narr_latlon.h5")
output:str = "html" # "json" "zip"


### FOD Program functions

testing individual functions

In [7]:
aws_config:dict = get_aws_config()
s3_client:S3Client = get_s3_client(aws_config)
s3_client

### Wind data

create the objects to using s3 directly from the classes

In [8]:
grid_index = GridIndexS3(narr_grid_file= narr_grid_file, bucket=narr_bucket, s3_client= s3_client)
wind_data = WindDataS3(grid_index, bucket = narr_bucket, narr_data_dir=narr_data_dir)
WindData

Bucket mioffsetnarr exists
Bucket mioffsetnarr exists


mioffset.narr_data.WindData

**Alternative: Wind Data Class Factory**

generate the wind data from a factory using the types as inputs
this will allow setting the location and file formats as input params

In [9]:
# note the factory builds the GridIndex so requires the narr_grid_file param
wind_data:WindData = wind_data_factory(location = "S3", 
                      narr_grid_file = narr_grid_file, 
                      narr_data_dir = narr_data_dir,
                      narr_bucket = narr_bucket, 
                      s3_client = s3_client)  

Bucket mioffsetnarr exists
Bucket mioffsetnarr exists


Using this object, read Wind Data from S3 for our lat/long coordinates

In [10]:
narr_timeseries = wind_data.read_narr_timeseries_json(lat, lon, format = "FOD")
narr_timeseries

{'pc': array([5., 5., 5., ..., 4., 5., 4.], shape=(87600,)),
 'ws': array([6.787, 5.087, 5.152, ..., 2.176, 2.566, 2.281], shape=(87600,)),
 'wd': array([ 86.269,  86.532,  86.583, ..., 117.759, 100.491, 107.65 ],
       shape=(87600,))}

### More parameters

these are used in the 'full' FOD run but not here (for now).  Instead we are running for the full year and just the one time. 

In [ ]:
# # Use full dataset: 00 UCT 1 Jan to 21 UCT 31 Dec
ts,te=(0,2920)
# ignore time options, run for whole season

# narr_timeseries =  filter_narr_timeseries(narr_timeseries, ts, te)
# this is based on the "TIME_FLAG" parameter
topt = 1

### Run the model 
with these data and OEF 

In [ ]:
D = fod_model(pc=narr_timeseries['pc'], wind_speed=narr_timeseries['ws'], wind_direction=narr_timeseries['wd'], odor_index=odor_index)
D 

### Table Test
formatted table to text file from model output


In [ ]:
if(topt == 1):            
    text_file_name:str = 'table_setbackdistance_FY'            
elif(topt == 2):
    text_file_name:str = 'table_setbackdistance_WS' 
else:
    text_file_name:str = 'table_setbackdistance'

table_text = setback_text_table(D)
from pprint import pprint
pprint(table_text)

### Plot test

create a plot, convert to SVG and display here

In [ ]:
p = fod3.footprint_plots(D, E = odor_index, topt = topt)
svg1 = fod3.matplotlib_to_svg(p)
svg1

from IPython.display import SVG, display
SVG(data = svg1)

## Map Test

make a Lat/Lon array, used by both KML and shape file writers
this is not the same as 2018 version:
 b/c uses geodesic fn since vincenty was deprecated

In [ ]:
LL = fod_plot_to_ll(D, lat, lon)

### KML

this is in the original code but we may not need it for web app which does the mapping

In [ ]:
import simplekml
if(topt == 2):
    kml_file_name = "kml_footprint_WS.kml" 
else:
    kml_file_name = "kml_footprint_FY.kml"
    
kml:simplekml.Kml = fod_kml(LL, E=odor_index, lat=lat, lon=lon)

def kml2base64(kml:str|simplekml.Kml, kml_file_name="kml"):
    if hasattr(kml, 'kml'):
        kml_xml:str = kml.kml()
    elif isinstance(kml, str):
        kml_xml:str = kml        
    else:
        # don't know what this is
        raise RuntimeError("kml sent to kml2base64 is not a recognized type (kml or xml str)")
    kmlb64 = base64.urlsafe_b64encode(kml_xml.encode('utf-8'))
    kml_dict = {kml_file_name:kmlb64}
    return kml_dict

### GeoJSON for mapping

GeoJSON map format can be used immediately 


In [ ]:
geo_json = fod_geojson(LL, E= odor_index, lat = lat, lon = lon)
geo_json

### create a map! 

use folium to make a map. Folium is not a requirement for this package, 
but 

In [ ]:
try:
    # Attempt to import the package
    __import__('folium') # Use __import__ for dynamic name checking
    # Or directly: import requests
    m = folium.Map(tiles="cartodbpositron")
    folium.GeoJson(geo_json, name="Setback distances").add_to(m)
    folium.LayerControl().add_to(m)   
    m 
except ModuleNotFoundError:
    # Handle the case where the module is not found
    print(f"folium is not installed.")
    

In [ ]:
from mioffset.aws import check_bucket, S3Client

wind_data = wind_data_factory(location = "S3", 
                      narr_grid_file = narr_grid_file, 
                      narr_data_dir = narr_data_dir,
                      narr_bucket = narr_bucket, 
                      s3_client = s3_client)   

## from model to output strategy 


- setup the config based on inputs (module)
    - python interface using pydantic
    - cli interface using typer
    - maybe http interface using fastapi (or just lambda)
    - use CLI but also default to env using arg parse or typer
- data
- fod "runner"
    - get and checks inputs (some, they should already be checked)
    - use typing py

- fod_output: use a class and subclasses for each format??
    - for code re-use only since they are run-once and quit
    - every method needs wind_data 
    - is running the model the responsibility of the class, or external function?
    - external model runner is testable

- do the classes make the things or just assemble them?
    - I think they make the things using the functions they need
    - this will allow a user of the class to make plots without caring the destination


Starter functions to: 
1. run using JSON data stored in S3
2. create a JSON response object for an API

In [ ]:
def fod_run_s3(lat, lon, odor_index,narr_grid_file, narr_data_dir,narr_bucket, s3_client):    
    wind_data:WindData = wind_data_factory(location = "S3", 
                      narr_grid_file = narr_grid_file, 
                      narr_data_dir = narr_data_dir,
                      narr_bucket = narr_bucket, 
                      s3_client = s3_client)    
    wd = wind_data.read_narr_timeseries_json(latval = lat, lonval = lon, format = "FOD")
    D = fod3.fod_model(odor_index= odor_index,
                       pc = wd['pc'], wind_speed=wd['wind_speed'], wind_direction=wd['wind_direction']
                       )    
    return(D)

def build_fod_response(D, lat, lon, odor_index,pkg_version = "0.1")->dict:
    #TODO find method for extracting package version    
    # visualizatons prep
    #TODO move these to class or external, this should just be for resp prep
    # text
    table_text = setback_text_table(D)
    # plot
    plot = footprint_plots(D, E = odor_index, topt = topt)
    plot_svg = matplotlib_to_svg(plot)
    plot_svg_encoded = plot_svg.encode(encoding='utf-8')
    # map
    geo_json = fod_geojson(LL, E= odor_index, lat = lat, lon = lon)
    # prep response
    fod_response = {}
    fod_response['meta'] = {
        'version' : pkg_version,
        'timestamp' : datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }
    fod_response['inputs'] = {
        'lat':lat, 
        'lon':lon, 
        'oef':odor_index 
    }
    #TODO add names for captions
    fod_response['outputs'] = {
        'raw' : {
            'name': 'Raw Data', 
            'format': 'application/json',  
            'data': fod2dict(D) 
            },
        'table' : {
            'name': 'Table', 
            'format': 'text/plain',
            'data': table_text
            },
        'map' : {
            'name': 'Map', 
            'format' : 'application/geo+json', 
            'data' : geo_json 
            },
        'plot' : {
            'name': 'Plot', 
            'format' : 'image/svg+xml;charset=utf-8', 
            'data' :plot_svg_encoded 
            }                           
    }
                        
    fod_response_json:str = json.dumps(fod_response)
    return(fod_response_json)


In [ ]:
from mioffset.aws import check_bucket, get_aws_config, get_s3_client
s3_client = get_s3_client()
check_bucket(s3_client, bucket_name=narr_bucket)
D = fod_run_s3(lat, lon, odor_index,narr_grid_file, narr_data_dir,narr_bucket, s3_client)
mioffset_json = build_fod_response_json(D, lat, lon, odor_index)


### Save example outputs

Optionally Save to S3

In [ ]:

s3_client.put_object(
    Body=mioffset_json,  # Serialize the JSON object
    Bucket=narr_bucket,      # Specify your bucket name
    Key= 'outputs/fod_example_output_1.json'
)

optionally save to disk

In [ ]:
with open('example_fod_output.json', 'w') as f:
    f.writelines(mioffset_json)



In [ ]:
with open('example_fod_geojson.json', 'w') as f:
    json.dump(geo_json1, f)
    